In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

import os
import torch

root = '/content/gdrive/MyDrive'

required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P10.mat'
    for split in ('train', 'validation', 'test')
]
required += [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_{split}_P10.json'
    for split in ('validation', 'test')
]
required += [
    f'{root}/unfolded_dl_v8_continue_p10_results/model_P10_seed{seed}_v8.pt'
    for seed in (20261026, 20261027, 20261028)
]

missing = [path for path in required if not os.path.isfile(path)]
assert not missing, f'Missing Drive files: {missing}'

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU connected')
print('P=10 data, GS curves, and V8 checkpoints found.')

Mounted at /content/gdrive
GPU: No GPU connected
P=10 data, GS curves, and V8 checkpoints found.


In [2]:
from google.colab import files

uploaded = files.upload()

required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
    'colab_unfolded_dl_v7_continue_p10.py',
    'colab_unfolded_dl_v8_continue_p10.py',
    'colab_unfolded_dl_v9_highsnr_finetune.py',
}

missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All six scripts uploaded.')

Saving colab_unfolded_dl_v3_highsnr.py to colab_unfolded_dl_v3_highsnr.py
Saving colab_unfolded_dl_v5_snr_aware.py to colab_unfolded_dl_v5_snr_aware.py
Saving colab_unfolded_dl_v6_gs_gap.py to colab_unfolded_dl_v6_gs_gap.py
Saving colab_unfolded_dl_v7_continue_p10.py to colab_unfolded_dl_v7_continue_p10.py
Saving colab_unfolded_dl_v8_continue_p10.py to colab_unfolded_dl_v8_continue_p10.py
Saving colab_unfolded_dl_v9_highsnr_finetune.py to colab_unfolded_dl_v9_highsnr_finetune.py
All six scripts uploaded.


In [3]:
import torch

assert torch.cuda.is_available(), 'Connect a T4 GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))

AssertionError: Connect a T4 GPU runtime first.

In [4]:
!python /content/colab_unfolded_dl_v9_highsnr_finetune.py

Using device: cpu
P=10 seed=20261026 V9_epoch=001 worst validation DL-GS=+0.7012 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=002 worst validation DL-GS=+0.7005 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=003 worst validation DL-GS=+0.7000 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=004 worst validation DL-GS=+0.6994 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=005 worst validation DL-GS=+0.6988 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=006 worst validation DL-GS=+0.6983 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=007 worst validation DL-GS=+0.6978 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=008 worst validation DL-GS=+0.6972 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=009 worst validation DL-GS=+0.6968 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=010 worst validation DL-GS=+0.6964 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=011 worst validation DL-GS=+0.6959 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=012 worst validation DL-GS=+0.6955 dB lr=1.00e-05
P=10 seed=20261026 V9_epoch=013 worst validati

In [5]:
import json
import numpy as np
import torch

root = '/content/gdrive/MyDrive'
v8_dir = f'{root}/unfolded_dl_v8_continue_p10_results'
v9_dir = f'{root}/unfolded_dl_v9_highsnr_finetune_p10_results'

seeds = (20261026, 20261027, 20261028)
v8 = [
    torch.load(
        f'{v8_dir}/model_P10_seed{seed}_v8.pt',
        map_location='cpu',
        weights_only=False
    )['result']
    for seed in seeds
]

with open(f'{v9_dir}/summary_unfolded_v9_highsnr_p10.json', encoding='utf-8') as f:
    v9 = json.load(f)

snrs = [-5, 0, 5, 10, 15, 20, 25, 30]

print('SNR | V8 mean DL-GS | V9 mean DL-GS | V9 seeds beating GS')
for snr in snrs:
    key = str(snr)
    v8_gaps = [result['test_dl_minus_gs_db_by_snr'][key] for result in v8]
    v9_gaps = [result['test_dl_minus_gs_db_by_snr'][key] for result in v9]

    print(
        f'{snr:>3} dB | {np.mean(v8_gaps):+8.3f} dB | '
        f'{np.mean(v9_gaps):+8.3f} dB | '
        f'{sum(gap < 0 for gap in v9_gaps)}/3'
    )

print('V9 results folder:', v9_dir)

SNR | V8 mean DL-GS | V9 mean DL-GS | V9 seeds beating GS
 -5 dB |   -9.281 dB |   -9.286 dB | 3/3
  0 dB |   -5.966 dB |   -5.952 dB | 3/3
  5 dB |   -3.185 dB |   -3.129 dB | 3/3
 10 dB |   -1.145 dB |   -1.077 dB | 3/3
 15 dB |   -0.633 dB |   -0.584 dB | 3/3
 20 dB |   -0.101 dB |   -0.083 dB | 3/3
 25 dB |   +0.044 dB |   +0.051 dB | 0/3
 30 dB |   +0.590 dB |   +0.564 dB | 0/3
V9 results folder: /content/gdrive/MyDrive/unfolded_dl_v9_highsnr_finetune_p10_results
